In [11]:
import zipfile
import os
import pandas as pd

In [12]:
!git clone https://github.com/manikantavs01/DeepLearning_Hackaton.git


fatal: destination path 'DeepLearning_Hackaton' already exists and is not an empty directory.


In [13]:
# Step 3: Extract the zip file
# Create a directory for extraction
extract_dir = "/content/DeepLearning_Hackaton/train"
os.makedirs(extract_dir, exist_ok=True)

# Open and extract the zip file
with zipfile.ZipFile("/content/DeepLearning_Hackaton/train.zip", 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Contents extracted to {extract_dir}")

Contents extracted to /content/DeepLearning_Hackaton/train


In [14]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.utils.data import random_split
from sklearn.model_selection import train_test_split
from PIL import Image


# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
device

device(type='cpu')

In [17]:
# Define transformations for the training and test datasets
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet mean/std
])

In [18]:
train=pd.read_csv("/content/DeepLearning_Hackaton/train/train.csv")
test=pd.read_csv("/content/DeepLearning_Hackaton/sample_submission.csv")

In [19]:
test.shape

(706, 2)

In [20]:
train_paths=train['image_names'].values
train_emer=train['emergency_or_not'].values
test_paths=test['image_names'].values
test_emer=test['emergency_or_not'].values

In [22]:
# Function to load and preprocess images
def load_images(file_paths, labels, transform):
    images = []
    directory = '/content/DeepLearning_Hackaton/train/images'
    for file_path in file_paths:
        file_path = os.path.join(directory, str(file_path))
        image = Image.open(file_path).convert('RGB')
        image = transform(image)
        images.append(image)
    return torch.stack(images), torch.tensor(labels)

In [23]:
# Load datasets
train_images, train_labels = load_images(train_paths, train_emer, transform)
test_images, test_labels = load_images(test_paths, test_emer, transform)
train_dataset = TensorDataset(train_images, train_labels)
test_dataset = TensorDataset(test_images, test_labels)
#train_dataset = datasets.ImageFolder(root='/content/DeepLearning_Hackaton/train', transform=train_transforms)
#test_dataset = datasets.ImageFolder(root='/content/test/test', transform=test_transforms)

In [25]:
len(test_dataset)

706

In [26]:
# Split training data into training and validation sets
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

In [67]:
len(val_data)

330

In [75]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [79]:
len(val_loader)

11

In [76]:
# Define the CNN model architecture (no modular code)
conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
fc1 = nn.Linear(128 * 16 * 16, 512)
fc2 = nn.Linear(512, 1)  # Binary
dropout = nn.Dropout(0.5)

In [80]:
# Initialize the model
model = nn.Sequential(
    conv1,
    nn.ReLU(),
    pool,
    conv2,
    nn.ReLU(),
    pool,
    conv3,
    nn.ReLU(),
    pool,
    nn.Flatten(),
    fc1,
    nn.ReLU(),
    dropout,
    fc2
).to(device)


In [82]:
# Define the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
optimizer = optim.Adam(model.parameters(), lr=0.001)

def calculate_accuracy(outputs, labels):
    preds = torch.round(torch.sigmoid(outputs))
    correct = (preds == labels).float()
    accuracy = correct.sum() / len(correct)
    return accuracy

In [123]:
# Training the model
num_epochs = 5
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    running_accuracy = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        labels = labels.float().unsqueeze(1)  # Ensure labels are the correct shape
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Track the loss and accuracy
        running_loss += loss.item()
        running_accuracy += calculate_accuracy(outputs, labels).item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = running_accuracy / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%')

Epoch [1/5], Loss: 0.0106, Accuracy: 1.00%
Epoch [2/5], Loss: 0.0107, Accuracy: 1.00%
Epoch [3/5], Loss: 0.0081, Accuracy: 1.00%
Epoch [4/5], Loss: 0.0091, Accuracy: 1.00%
Epoch [5/5], Loss: 0.0073, Accuracy: 1.00%


In [132]:
labels

tensor([[0.],
        [1.],
        [0.],
        [1.]])

In [133]:
# Evaluate the model on the validation set
model.eval()  # Set the model to evaluation mode

running_val_accuracy = 0
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        labels = labels.float().unsqueeze(1)  # Ensure labels are the correct shape
        running_val_accuracy += calculate_accuracy(outputs, labels).item()

val_accuracy = 100*running_val_accuracy / len(val_loader)
print(f'Validation Accuracy: {val_accuracy:.2f}%')

Validation Accuracy: 81.93%


In [108]:
# Evaluate on the test set
model.eval()
running_test_accuracy = 0.0
output_test= []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        labels = labels.float().unsqueeze(1)
        output_test.append(torch.round(torch.sigmoid(outputs)))
        running_test_accuracy += calculate_accuracy(outputs, labels).item()

test_accuracy = 100 * running_test_accuracy / len(test_loader)
print(f'Test Accuracy: {test_accuracy:.2f}%')


Test Accuracy: 61.82%


In [122]:
output_test[0].size()
flattened_tensors = [tensor.view(-1, 1) for tensor in output_test]
single_column_tensor = torch.cat(flattened_tensors, dim=0)

# Convert the tensor to a DataFrame
df = pd.DataFrame(single_column_tensor.numpy(), columns=['emergency_or_not'])
# Combine the selected columns into a new DataFrame
combined_df = pd.DataFrame({'Image_Names': test['image_names'], 'emergency_or_not': df['emergency_or_not']})

# Write the DataFrame to an Excel file
combined_df.to_excel('test.xlsx', index=False)

print("List of tensors has been written to tensor_list_output.xlsx")

List of tensors has been written to tensor_list_output.xlsx
